# AI Studio LLM Evaluation Results Analysis

This notebook analyzes the `result.json` file containing LLM evaluation results for agent execution traces.

## Key Metrics to Analyze:
- Answer found rate
- Step index distribution
- Worker performance
- Tool usage patterns
- Failure analysis

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from pathlib import Path

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Configure display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print('Libraries loaded successfully')

## 1. Load Data

In [ ]:
# Load result.json
result_file = Path('result.json')

with open(result_file, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

print(f'Loaded {len(raw_data)} trace results')
print(f'Sample trace ID: {raw_data[0]["trace_id"]}')

In [ ]:
# Convert to structured DataFrame
records = []

for item in raw_data:
    trace_id = item.get('trace_id', '')
    answer_found = item.get('answer_found', {})
    
    step_index = answer_found.get('step_index', -1)
    worker = answer_found.get('worker', 'Unknown')
    evidence = answer_found.get('evidence', '')
    final_judgement = item.get('final_judgement', '')
    
    tools_involved = answer_found.get('tools_involved', [])
    tool_names = [t.get('tool', '') for t in tools_involved]
    tool_steps = [t.get('step', 0) for t in tools_involved]
    tool_reasons = [t.get('reason', '') for t in tools_involved]
    
    num_tools = len(tools_involved)
    
    found = step_index >= 0
    
    records.append({
        'trace_id': trace_id,
        'step_index': step_index,
        'worker': worker,
        'found': found,
        'num_tools': num_tools,
        'tools': tool_names,
        'tool_steps': tool_steps,
        'evidence': evidence,
        'final_judgement': final_judgement,
        'evidence_length': len(evidence),
        'judgement_length': len(final_judgement)
    })

df = pd.DataFrame(records)

print(f'DataFrame shape: {df.shape}')
df.head(10)

## 2. Overall Summary Statistics

In [ ]:
# Overall statistics
total = len(df)
found_count = df['found'].sum()
not_found_count = total - found_count

print('=' * 60)
print('OVERALL SUMMARY')
print('=' * 60)
print(f'Total traces: {total}')
print(f'Answers found: {found_count} ({found_count/total*100:.1f}%)
print(f'Answers NOT found: {not_found_count} ({not_found_count/total*100:.1f}%)')
print('=' * 60)

# Create summary visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart for found/not found
colors = ['#2ecc71', '#e74c3c']
axes[0].pie([found_count, not_found_count], 
            labels=['Found', 'Not Found'],
            autopct='%1.1f%%',
            colors=colors,
            explode=(0.05, 0.05),
            shadow=True,
            startangle=90)
axes[0].set_title('Answer Found Rate', fontsize=14, fontweight='bold')

# Bar chart
axes[1].bar(['Found', 'Not Found'], [found_count, not_found_count], color=colors)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Distribution of Results', fontsize=14, fontweight='bold')
for i, v in enumerate([found_count, not_found_count]):
    axes[1].text(i, v + 1, str(v), ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('summary_overall.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Step Index Analysis

In [ ]:
# Filter only found answers
df_found = df[df['found'] == True].copy()

print(f'Analyzing {len(df_found)} traces where answer was found')
print(f'\nStep Index Statistics:')
print(df_found['step_index'].describe())

# Calculate percentiles
percentiles = [10, 25, 50, 75, 90, 95]
print(f'\nPercentiles:')
for p in percentiles:
    val = np.percentile(df_found['step_index'], p)
    print(f'  {p}th percentile: {val:.0f}')

In [ ]:
# Step index distribution visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Histogram
axes[0, 0].hist(df_found['step_index'], bins=30, color='#3498db', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(df_found['step_index'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df_found["step_index"].mean():.1f}')
axes[0, 0].axvline(df_found['step_index'].median(), color='orange', linestyle='--', linewidth=2, label=f'Median: {df_found["step_index"].median():.0f}')
axes[0, 0].set_xlabel('Step Index', fontsize=12)
axes[0, 0].set_ylabel('Frequency', fontsize=12)
axes[0, 0].set_title('Distribution of Step Indices', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Box plot
axes[0, 1].boxplot(df_found['step_index'], vert=True, patch_artist=True)
axes[0, 1].set_ylabel('Step Index', fontsize=12)
axes[0, 1].set_title('Step Index Box Plot', fontsize=14, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Step count bar chart
step_counts = df_found['step_index'].value_counts().sort_index()
axes[1, 0].bar(step_counts.index, step_counts.values, color='#9b59b6', edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Step Index', fontsize=12)
axes[1, 0].set_ylabel('Number of Traces', fontsize=12)
axes[1, 0].set_title('Step Index Frequency', fontsize=14, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Cumulative distribution
sorted_steps = np.sort(df_found['step_index'])
cumulative = np.arange(1, len(sorted_steps) + 1) / len(sorted_steps)
axes[1, 1].plot(sorted_steps, cumulative, color='#e67e22', linewidth=2)
axes[1, 1].fill_between(sorted_steps, cumulative, alpha=0.3, color='#e67e22')
axes[1, 1].set_xlabel('Step Index', fontsize=12)
axes[1, 1].set_ylabel('Cumulative Proportion', fontsize=12)
axes[1, 1].set_title('Cumulative Distribution', fontsize=14, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

# Add percentile markers
for p in [25, 50, 75]:
    val = np.percentile(df_found['step_index'], p)
    axes[1, 1].axvline(val, color='red', linestyle=':', alpha=0.7)
    axes[1, 1].text(val, p/100, f'{p}%', fontsize=10)

plt.tight_layout()
plt.savefig('step_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Categorize by performance tier
def categorize_performance(step):
    if step <= 20:
        return 'Fast (≤20 steps)'
    elif step <= 30:
        return 'Normal (21-30 steps)'
    elif step <= 40:
        return 'Slow (31-40 steps)'
    else:
        return 'Very Slow (>40 steps)'

df_found['performance_tier'] = df_found['step_index'].apply(categorize_performance)

tier_counts = df_found['performance_tier'].value_counts()

print('Performance Tier Distribution:')
print(tier_counts)
print(f'\nPercentage breakdown:')
for tier, count in tier_counts.items():
    print(f'  {tier}: {count} ({count/len(df_found)*100:.1f}%)')

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))

colors_tiers = ['#27ae60', '#3498db', '#f39c12', '#e74c3c']
tier_order = ['Fast (≤20 steps)', 'Normal (21-30 steps)', 'Slow (31-40 steps)', 'Very Slow (>40 steps)']

bars = ax.bar(tier_order, [tier_counts.get(t, 0) for t in tier_order], color=colors_tiers, edgecolor='black')

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_xlabel('Performance Tier', fontsize=12)
ax.set_ylabel('Number of Traces', fontsize=12)
ax.set_title('Performance Tier Distribution', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('performance_tiers.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Worker Analysis

In [ ]:
# Worker distribution
worker_counts = df['worker'].value_counts()

print('Worker Distribution (All Traces):')
print(worker_counts.head(15))

# Worker performance
worker_stats = df.groupby('worker').agg({
    'found': ['sum', 'count', 'mean'],
    'step_index': ['mean', 'min', 'max', 'std']
}).round(2)

worker_stats.columns = ['found_count', 'total_count', 'found_rate', 'avg_step', 'min_step', 'max_step', 'std_step']
worker_stats = worker_stats.sort_values('total_count', ascending=False)

print(f'\nWorker Performance Summary:')
print(worker_stats.head(15))

In [ ]:
# Worker visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 10 workers by count
top_workers = worker_counts.head(10)
axes[0].barh(top_workers.index[::-1], top_workers.values[::-1], color='#8e44ad', edgecolor='black')
axes[0].set_xlabel('Number of Traces', fontsize=12)
axes[0].set_title('Top 10 Workers by Frequency', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')

# Worker found rate
top_worker_stats = worker_stats.head(10)
axes[1].barh(top_worker_stats.index[::-1], top_worker_stats['found_rate'][::-1] * 100, 
             color='#16a085', edgecolor='black')
axes[1].set_xlabel('Found Rate (%)', fontsize=12)
axes[1].set_title('Top 10 Workers - Answer Found Rate', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')
axes[1].axvline(87.1, color='red', linestyle='--', linewidth=2, label='Overall Rate (87.1%)')
axes[1].legend()

plt.tight_layout()
plt.savefig('worker_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Worker vs Step Index scatter
fig, ax = plt.subplots(figsize=(14, 8))

# Get top workers for coloring
top_worker_names = worker_counts.head(8).index.tolist()

# Create scatter plot
for i, worker in enumerate(top_worker_names):
    worker_data = df_found[df_found['worker'] == worker]
    ax.scatter(worker_data['step_index'], [i] * len(worker_data), 
               s=100, alpha=0.7, label=worker)

# Other workers
other_data = df_found[~df_found['worker'].isin(top_worker_names)]
ax.scatter(other_data['step_index'], [len(top_worker_names)] * len(other_data),
           s=100, alpha=0.5, c='gray', label='Other')

ax.set_xlabel('Step Index', fontsize=12)
ax.set_ylabel('Worker', fontsize=12)
ax.set_yticks(range(len(top_worker_names) + 1))
ax.set_yticklabels(top_worker_names + ['Other'])
ax.set_title('Step Index Distribution by Worker', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('worker_step_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Tool Usage Analysis

In [ ]:
# Extract all tools
all_tools = []
for tools_list in df['tools']: 
    all_tools.extend(tools_list)

tool_counts = Counter(all_tools)

print('Tool Usage Summary:')
print(f'Total tool invocations: {len(all_tools)}')
print(f'Unique tools: {len(tool_counts)}')
print(f'\nTop 15 Most Used Tools:')
for tool, count in tool_counts.most_common(15):
    print(f'  {tool}: {count}')

In [ ]:
# Tool visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Top 15 tools bar chart
top_tools = dict(tool_counts.most_common(15))
axes[0].barh(list(top_tools.keys())[::-1], list(top_tools.values())[::-1], 
             color='#2980b9', edgecolor='black')
axes[0].set_xlabel('Usage Count', fontsize=12)
axes[0].set_title('Top 15 Most Used Tools', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')

# Tool usage by number of tools per trace
axes[1].hist(df_found['num_tools'], bins=range(0, max(df_found['num_tools']) + 2), 
             color='#e74c3c', edgecolor='black', alpha=0.7, align='left')
axes[1].set_xlabel('Number of Tools Used', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Number of Tools Used per Trace', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Add statistics
avg_tools = df_found['num_tools'].mean()
axes[1].axvline(avg_tools, color='blue', linestyle='--', linewidth=2, label=f'Mean: {avg_tools:.1f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('tool_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Tool effectiveness analysis
# For each tool, calculate: how often it appears in successful vs failed traces

tool_success_rate = defaultdict(lambda: {'success': 0, 'total': 0})

for _, row in df.iterrows():
    for tool in row['tools']:
        tool_success_rate[tool]['total'] += 1
        if row['found']:
            tool_success_rate[tool]['success'] += 1

# Create DataFrame
tool_effectiveness = pd.DataFrame([
    {'tool': tool, 'success': data['success'], 'total': data['total'], 
     'rate': data['success'] / data['total'] * 100 if data['total'] > 0 else 0}
    for tool, data in tool_success_rate.items()
]).sort_values('total', ascending=False)

print('Tool Effectiveness (appears in successful traces):')
print(tool_effectiveness.head(15))

# Visualization
fig, ax = plt.subplots(figsize=(12, 8))

top_eff = tool_effectiveness.head(12)
x = range(len(top_eff))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], top_eff['success'], width, label='Success', color='#27ae60')
bars2 = ax.bar([i + width/2 for i in x], top_eff['total'] - top_eff['success'], width, label='Failed', color='#e74c3c')

ax.set_xlabel('Tool', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Tool Success vs Failure Distribution', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(top_eff['tool'], rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('tool_effectiveness.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Failure Analysis

In [ ]:
# Analyze failed traces
df_failed = df[df['found'] == False].copy()

print(f'Analyzing {len(df_failed)} failed traces')
print(f'\nFailed Trace IDs:')
for tid in df_failed['trace_id'].values:
    print(f'  {tid}')

In [ ]:
# Examine failure reasons
print('Failure Evidence Analysis:')
print('=' * 80)

for _, row in df_failed.iterrows():
    print(f'\nTrace: {row["trace_id"]}')
    print(f'Worker: {row["worker"]}')
    print(f'Num tools used: {row["num_tools"]}')
    print(f'Evidence (first 200 chars): {row["evidence"][:200]}...')
    print('-' * 80)

In [ ]:
# Common failure themes
failure_keywords = ['no relevant', 'not found', 'missing', 'failed', 'no content', 
                     'not present', 'no match', 'unavailable', 'could not']

keyword_counts = Counter()
for evidence in df_failed['evidence']:
    evidence_lower = evidence.lower()
    for keyword in failure_keywords:
        if keyword in evidence_lower:
            keyword_counts[keyword] += 1

print('Common Failure Keywords in Evidence:')
for keyword, count in keyword_counts.most_common():
    print(f'  "{keyword}" appears in {count} failures')

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))

if keyword_counts:
    ax.barh(list(keyword_counts.keys())[::-1], list(keyword_counts.values())[::-1],
            color='#c0392b', edgecolor='black')
    ax.set_xlabel('Frequency', fontsize=12)
    ax.set_title('Common Failure Keywords', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('failure_keywords.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Worker distribution for failed traces
failed_worker_counts = df_failed['worker'].value_counts()

print('Worker Distribution in Failed Traces:')
print(failed_worker_counts)

# Compare with overall worker distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Failed workers
axes[0].barh(failed_worker_counts.index[::-1], failed_worker_counts.values[::-1],
             color='#e74c3c', edgecolor='black')
axes[0].set_xlabel('Failed Count', fontsize=12)
axes[0].set_title('Workers with Failed Traces', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')

# Failure rate by worker
worker_failure_rate = df.groupby('worker')['found'].apply(lambda x: (1 - x.mean()) * 100).sort_values(ascending=False)
top_failure_workers = worker_failure_rate[worker_failure_rate > 0].head(10)

axes[1].barh(top_failure_workers.index[::-1], top_failure_workers.values[::-1],
             color='#d35400', edgecolor='black')
axes[1].set_xlabel('Failure Rate (%)', fontsize=12)
axes[1].set_title('Workers with Highest Failure Rate', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')
axes[1].axvline(12.9, color='red', linestyle='--', linewidth=2, label='Overall (12.9%)')
axes[1].legend()

plt.tight_layout()
plt.savefig('failure_workers.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Correlation Analysis

In [ ]:
# Correlation between metrics
correlation_data = df_found[['step_index', 'num_tools', 'evidence_length', 'judgement_length']].copy()

corr_matrix = correlation_data.corr()

print('Correlation Matrix:')
print(corr_matrix.round(3))

# Heatmap
fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=2, fmt='.2f', ax=ax)
ax.set_title('Correlation Heatmap', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Scatter plots for correlations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Step vs Num Tools
axes[0, 0].scatter(df_found['step_index'], df_found['num_tools'], alpha=0.6, c='#3498db', s=50)
axes[0, 0].set_xlabel('Step Index', fontsize=12)
axes[0, 0].set_ylabel('Number of Tools', fontsize=12)
axes[0, 0].set_title('Step Index vs Number of Tools', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Step vs Evidence Length
axes[0, 1].scatter(df_found['step_index'], df_found['evidence_length'], alpha=0.6, c='#e74c3c', s=50)
axes[0, 1].set_xlabel('Step Index', fontsize=12)
axes[0, 1].set_ylabel('Evidence Length (chars)', fontsize=12)
axes[0, 1].set_title('Step Index vs Evidence Length', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Num Tools vs Evidence Length
axes[1, 0].scatter(df_found['num_tools'], df_found['evidence_length'], alpha=0.6, c='#27ae60', s=50)
axes[1, 0].set_xlabel('Number of Tools', fontsize=12)
axes[1, 0].set_ylabel('Evidence Length (chars)', fontsize=12)
axes[1, 0].set_title('Number of Tools vs Evidence Length', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Step vs Judgement Length
axes[1, 1].scatter(df_found['step_index'], df_found['judgement_length'], alpha=0.6, c='#9b59b6', s=50)
axes[1, 1].set_xlabel('Step Index', fontsize=12)
axes[1, 1].set_ylabel('Judgement Length (chars)', fontsize=12)
axes[1, 1].set_title('Step Index vs Judgement Length', fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('correlation_scatters.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Detailed Trace Analysis

In [ ]:
# Top 10 fastest traces
fastest_traces = df_found.nsmallest(10, 'step_index')[['trace_id', 'step_index', 'worker', 'num_tools', 'tools']]

print('Top 10 Fastest Traces (earliest step where answer found):')
print('=' * 80)
for _, row in fastest_traces.iterrows():
    print(f'{row["trace_id"]}: Step {row["step_index"]} | Worker: {row["worker"]} | Tools: {row["num_tools"]}')
    print(f'   Tools used: {row["tools"]}')
    print('-' * 80)

In [ ]:
# Top 10 slowest traces
slowest_traces = df_found.nlargest(10, 'step_index')[['trace_id', 'step_index', 'worker', 'num_tools', 'tools']]

print('Top 10 Slowest Traces (latest step where answer found):')
print('=' * 80)
for _, row in slowest_traces.iterrows():
    print(f'{row["trace_id"]}: Step {row["step_index"]} | Worker: {row["worker"]} | Tools: {row["num_tools"]}')
    print(f'   Tools used: {row["tools"]}')
    print('-' * 80)

In [ ]:
# Tool combinations analysis
df_found['tool_combo'] = df_found['tools'].apply(lambda x: tuple(sorted(set(x))))

combo_counts = df_found['tool_combo'].value_counts()

print('Most Common Tool Combinations:')
print('=' * 80)
for combo, count in combo_counts.head(10).items():
    print(f'{count} traces used: {list(combo)}')

## 9. Summary Statistics Table

In [ ]:
# Create comprehensive summary table
summary_stats = {
    'Metric': [
        'Total Traces',
        'Answers Found',
        'Answers Not Found',
        'Found Rate (%)',
        'Avg Step Index (found)',
        'Min Step Index',
        'Max Step Index',
        'Median Step Index',
        'Std Step Index',
        'Avg Tools per Trace',
        'Most Common Tool',
        'Most Common Worker',
        'Fastest Tier (≤20)',
        'Normal Tier (21-30)',
        'Slow Tier (31-40)',
        'Very Slow Tier (>40)'
    ],
    'Value': [
        len(df),
        found_count,
        not_found_count,
        f'{found_count/len(df)*100:.1f}',
        f'{df_found["step_index"].mean():.1f}',
        df_found['step_index'].min(),
        df_found['step_index'].max(),
        df_found['step_index'].median(),
        f'{df_found["step_index"].std():.1f}',
        f'{df_found["num_tools"].mean():.1f}',
        tool_counts.most_common(1)[0][0] if tool_counts else 'N/A',
        worker_counts.most_common(1)[0][0] if worker_counts else 'N/A',
        tier_counts.get('Fast (≤20 steps)', 0),
        tier_counts.get('Normal (21-30 steps)', 0),
        tier_counts.get('Slow (31-40 steps)', 0),
        tier_counts.get('Very Slow (>40 steps)', 0)
    ]
}

summary_df = pd.DataFrame(summary_stats)
print('\nCOMPREHENSIVE SUMMARY STATISTICS:')
print('=' * 50)
print(summary_df.to_string(index=False))

## 10. Export Results

In [ ]:
# Export analysis results
output_dir = Path('analysis_output')
output_dir.mkdir(exist_ok=True)

# Save processed DataFrame
df.to_csv(output_dir / 'processed_results.csv', index=False)
df_found.to_csv(output_dir / 'found_results.csv', index=False)
df_failed.to_csv(output_dir / 'failed_results.csv', index=False)

# Save summary
summary_df.to_csv(output_dir / 'summary_statistics.csv', index=False)

# Save worker stats
worker_stats.to_csv(output_dir / 'worker_statistics.csv')

# Save tool effectiveness
tool_effectiveness.to_csv(output_dir / 'tool_effectiveness.csv', index=False)

print(f'All results exported to: {output_dir}')
print(f'Files created:')
for f in output_dir.glob('*.csv'):
    print(f'  {f.name}')

In [ ]:
# Final summary
print('\n' + '=' * 80)
print('ANALYSIS COMPLETE')
print('=' * 80)
print(f'\nKey Findings:')
print(f'1. {found_count/len(df)*100:.1f}% of traces successfully found answers')
print(f'2. Average step to find answer: {df_found["step_index"].mean():.1f}')
print(f'3. Most effective tool: {tool_counts.most_common(1)[0][0]} ({tool_counts.most_common(1)[0][1]} uses)')
print(f'4. Most active worker: {worker_counts.most_common(1)[0][0]} ({worker_counts.most_common(1)[0][1]} traces)')
print(f'5. {len(df_failed)} traces failed - primarily due to missing content in corpus')
print(f'\nVisualizations saved as PNG files')
print(f'Data exports saved to analysis_output/')